# FADE — VLM Extraction Demo

**Runtime required:** GPU (T4 or better)  
In Colab: `Runtime → Change runtime type → T4 GPU`

**What this notebook does:**
1. Installs dependencies
2. **Section A** — Zero-shot inference test (~10 min)
3. **Section B** — Short QLoRA training proof-of-concept (~30–60 min on T4, ~15 min on A100)

## Setup

The repo is private. Before running this cell:
1. Click the **key icon** (Secrets) in the Colab left sidebar
2. Add a secret named `GITHUB_TOKEN` with a GitHub Personal Access Token
   - Generate one at: GitHub → Settings → Developer settings → Personal access tokens → Tokens (classic) → repo scope
3. Toggle **"Notebook access"** on for that secret

In [ ]:
import os
from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')
repo_url = f"https://{token}@github.com/juweriya1/ai-document-processing.git"

if not os.path.exists('ai-document-processing'):
    !git clone {repo_url} --quiet
    print("Cloned successfully.")
else:
    print("Repo already exists, pulling latest...")

%cd ai-document-processing
!git pull origin main --quiet
print("Ready.")

In [ ]:
# Install dependencies (~3-5 min)
!pip install -q \
    "transformers>=4.45.0" \
    torch torchvision \
    "bitsandbytes>=0.43.0" \
    "peft>=0.11.0" \
    accelerate \
    "qwen-vl-utils>=0.0.8" \
    "datasets>=2.18.0" \
    Pillow \
    trl
print("Dependencies installed.")

In [ ]:
# Verify GPU
import torch
assert torch.cuda.is_available(), "No GPU found — change runtime to T4 GPU."
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## Section A — Zero-shot Inference (~10 min)

Loads Qwen2-VL-7B-Instruct in 4-bit and runs extraction on a CORD receipt.  
No training needed.

In [ ]:
from datasets import load_dataset
import json

SAMPLE_INDEX = 0  # change to try different receipts (0–99)

print("Downloading one CORD test sample...")
dataset = load_dataset("naver-clova-ix/cord-v2", split="test")
row = dataset[SAMPLE_INDEX]
pil_image = row["image"]

gt_raw = json.loads(row["ground_truth"])["gt_parse"]
ground_truth = {
    "vendor_name":  gt_raw.get("store_info", {}).get("store_name"),
    "total_amount": gt_raw.get("total", {}).get("total_price"),
    "subtotal":     gt_raw.get("total", {}).get("subtotal_price"),
    "tax":          gt_raw.get("total", {}).get("tax_price"),
}

print(f"Image size: {pil_image.size}")
print("\nGround truth:")
for k, v in ground_truth.items():
    if v: print(f"  {k}: {v}")

pil_image

In [ ]:
import sys, time
sys.path.insert(0, '.')

from src.backend.extraction.vlm_extractor import VLMExtractor

print("Loading VLMExtractor (downloads ~15 GB on first run, ~3-5 min)...")
t0 = time.time()
extractor = VLMExtractor()
_ = extractor.model
print(f"Model loaded in {time.time()-t0:.0f}s")

print("\nRunning inference...")
t1 = time.time()
fields, line_items = extractor.extract_from_image(pil_image.convert("RGB"))
print(f"Done in {time.time()-t1:.1f}s")

In [ ]:
print("=" * 55)
print("EXTRACTED FIELDS")
print("=" * 55)
if fields:
    for f in fields:
        print(f"  {f['field_name']:<20} {str(f['field_value']):<25}  conf={f['confidence']:.2f}")
else:
    print("  (no fields extracted)")

if line_items:
    print("\nLINE ITEMS")
    print("=" * 55)
    for item in line_items:
        print(f"  {item.get('description',''):<30} qty={item.get('quantity',0)}  total={item.get('total',0)}")

print("\nCOMPARISON vs GROUND TRUTH")
print("=" * 55)
pred_map = {f["field_name"]: f["field_value"] for f in fields}
for field, gold in ground_truth.items():
    if not gold: continue
    pred = pred_map.get(field, "(missing)")
    match = "✓" if str(gold).strip() == str(pred).strip() else "✗"
    print(f"  {match} {field:<20} gold={gold}  pred={pred}")

---
## Section B — QLoRA Training Proof-of-Concept

Runs **100 steps** to confirm training works end-to-end.  
**Not the full run** — do the 500-step run on the A100 MIG.

| Hardware | 100 steps | 500 steps (full) |
|----------|-----------|------------------|
| T4 (Colab free) | ~45 min | ~3.5 hr |
| A100 40GB (Colab Pro) | ~15 min | ~1 hr |

**Skip this section if you only wanted to test inference.**

In [ ]:
# Free VRAM from inference before training
import gc
del extractor
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM reserved: {torch.cuda.memory_reserved(0)/1e9:.1f} GB")

In [ ]:
!python scripts/train_qlora.py \
    --max_steps 100 \
    --output_dir adapters/qlora_poc

print("\nAdapter saved to adapters/qlora_poc/")

In [ ]:
# Compare zero-shot vs fine-tuned on the same sample
ft_extractor = VLMExtractor(adapter_path="adapters/qlora_poc")
fields_ft, _ = ft_extractor.extract_from_image(pil_image.convert("RGB"))

pred_ft = {f["field_name"]: f["field_value"] for f in fields_ft}
print("FINE-TUNED (100 steps) vs GROUND TRUTH")
print("=" * 55)
for field, gold in ground_truth.items():
    if not gold: continue
    ft = pred_ft.get(field, "(missing)")
    match = "✓" if str(gold).strip() == str(ft).strip() else "✗"
    print(f"  {match} {field:<20} gold={gold}  pred={ft}")

In [ ]:
# Save adapter to Google Drive so it persists after session ends
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r adapters/qlora_poc "/content/drive/MyDrive/fade_adapters/"